# 9.0 - Active Domain Adaptation

The ADAPTATION axis: UCF101 is clean and centred, YouTube is not. This section pulls a few
YouTube clips per class, uses active learning to keep only the most informative ones, and
Reptile-adapts the head to them while replaying UCF exemplars so the original classes are
not forgotten.

State the limit up front: the ResNet is frozen, so low-level shift (compression, lighting,
resolution) cannot be adapted away. The LSTM and head can absorb a roughly linear feature
shift; they cannot recover what the conv layers discarded. This measures how much of the
gap the trainable part can close.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "PASTE_YOUR_KAGGLE_KEY"

## **Download DATASET**

UCF exemplars come from the cache; the frozen backbone turns YouTube clips into features.

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=False)

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Load Cache, Teacher Head, Backbone**

The winning CL head is the model being adapted. The ResNet backbone turns YouTube clips
into (16, 2048) features the same way UCF clips were cached.

In [ ]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
check_cache_labels(META, ALL_CLASSES_LIST)
class_to_idx = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}

In [ ]:
ADAPT_ARM = "replay_lwf"
head_path = f"{cfg.CKPT_DIR}/head_{ADAPT_ARM}.pt"
if not os.path.exists(head_path):
    raise FileNotFoundError(f"No head at {head_path}. Run notebook 4.6 first, or set ADAPT_ARM.")

teacher_head = make_temporal_head(len(ALL_CLASSES_LIST), lstm_state=LSTM_STATE, device=device)
teacher_head.load_state_dict(torch.load(head_path, map_location=device))
teacher_head.eval()
print(f"adapting head: {head_path}")

In [ ]:
cfg.RESNET50_PATH = f"{cfg.DRIVE_PROJECT}/resnet50/models/ResNet50_10C_groupsplit.pth"

teacher = ResNet50LSTMTeacher(hidden_size=cfg.TEACHER_HIDDEN, num_classes=len(cfg.SELECTED_CLASSES), dropout_p=cfg.DROPOUT_P).to(device)
mapped = remap_teacher_checkpoint(torch.load(cfg.RESNET50_PATH, map_location=device))
teacher.load_state_dict(mapped, strict=False)
for p in teacher.parameters():
    p.requires_grad_(False)
teacher.eval()
print("backbone frozen")

## **Download YouTube Clips**

One video per class from `cfg.YOUTUBE_CLIPS` (all base classes), cut into short clips with
the identical UCF transforms. If a download fails the class is skipped; the notebook stops
if nothing came through.

In [ ]:
!pip install -q yt-dlp

In [ ]:
clips_by_class, skipped = download_and_extract(
    cfg.YOUTUBE_CLIPS, cfg.YT_RAW_DIR, class_to_idx,
    clip_len=cfg.CLIP_LEN, n_clips=cfg.CLIPS_PER_VID,
)
if not clips_by_class:
    raise RuntimeError("No YouTube clips downloaded. Check the URLs in cfg.YOUTUBE_CLIPS and that yt-dlp works.")

## **Extract YouTube Features**

The frozen backbone over the YouTube clips, giving the same (16, 2048) features.

In [ ]:
yt_clip_list, yt_lab_list = [], []
for cls_name, clips in clips_by_class.items():
    for clip in clips:
        yt_clip_list.append(clip)
        yt_lab_list.append(class_to_idx[cls_name])

yt_x = torch.stack(yt_clip_list)
yt_y = torch.tensor(yt_lab_list, dtype=torch.long)
yt_loader = DataLoader(TensorDataset(yt_x, yt_y), batch_size=cfg.BATCH_SIZE, shuffle=False)

yt_s, yt_y = extract_frame_features(teacher.backbone, yt_loader, device, cfg.BACKBONE_DIM, torch.float32)
print(f"YouTube features: {tuple(yt_s.shape)}   classes {sorted(set(yt_lab_list))}")

## **Domain Shift**

Before adapting, measure how far each class moved. UCF and YouTube class prototypes are
compared in the head's 256-d space.

In [ ]:
ucf_emb, ucf_ey = head_embeddings(teacher_head, CACHE, [0], device, "test")
yt_emb_256      = teacher_head.embed(yt_s.to(device)).cpu()

ucf_stats = compute_class_stats(ucf_emb, ucf_ey)
yt_stats  = compute_class_stats(yt_emb_256, yt_y)
shifts    = measure_shift(ucf_stats, yt_stats, ALL_CLASSES_LIST)

## **Baseline (before adaptation)**

How the UCF-trained head does on YouTube, and on UCF base, before any adaptation.

In [ ]:
base_test_s, base_test_y = CACHE["t0_test"]

yt_acc_before  = eval_head(teacher_head, yt_s, yt_y, device)
ucf_acc_before = eval_head(teacher_head, base_test_s, base_test_y, device)
print(f"before | YouTube acc: {yt_acc_before:.2%} | UCF base acc: {ucf_acc_before:.2%}")

## **Active Learning**

Score every YouTube clip by predictive entropy and keep the most informative
`AL_BUDGET_PER_CLASS` per class. These are the clips worth "labelling".

In [ ]:
yt_by_class = {}
for cls_name in clips_by_class:
    idx = class_to_idx[cls_name]
    yt_by_class[cls_name] = yt_s[yt_y == idx]

selected = select_top_k(yt_by_class, cfg.AL_STRATEGY, cfg.AL_BUDGET_PER_CLASS,
                        head=teacher_head, device=device)

In [ ]:
al_list, al_ylist = [], []
for cls_name, idxs in selected.items():
    al_list.append(yt_by_class[cls_name][idxs])
    al_ylist.extend([class_to_idx[cls_name]] * len(idxs))

al_s = torch.cat(al_list)
al_y = torch.tensor(al_ylist, dtype=torch.long)
al_t = teacher_head.embed(al_s.to(device)).cpu()
print(f"AL-selected clips: {tuple(al_s.shape)}")

## **Reptile Adaptation**

Fill a replay buffer with UCF base exemplars (features and their teacher embeddings), then
Reptile-adapt a copy of the head on the AL clips mixed with replay. The replay is what
stops the UCF classes from being forgotten.

In [ ]:
buffer = SmartReplayBuffer(max_per_class=15)
base_s, base_yb = CACHE["t0_train"]
base_emb_all, _ = head_embeddings(teacher_head, CACHE, [0], device, "train")

for cls_name in cfg.SELECTED_CLASSES:
    idx  = class_to_idx[cls_name]
    mask = base_yb == idx
    buffer.add_class(idx, base_s[mask], base_emb_all[mask], strategy="hard")

print(f"buffer: {len(buffer.s_store)} classes")

In [ ]:
import copy
set_seed(cfg.SEED)
adapted = copy.deepcopy(teacher_head)

reptile_adapt(
    adapted, al_s, al_t, al_y, buffer, device,
    episodes=cfg.ADAPT_EPISODES, inner_lr=cfg.ADAPT_INNER_LR,
    inner_steps=cfg.ADAPT_INNER_STEPS, epsilon=cfg.ADAPT_EPSILON,
    mse_weight=0.5, eval_s=yt_s, eval_y=yt_y,
)

## **After adaptation**

In [ ]:
yt_acc_after  = eval_head(adapted, yt_s, yt_y, device)
ucf_acc_after = eval_head(adapted, base_test_s, base_test_y, device)

print(f"YouTube : {yt_acc_before:.2%} -> {yt_acc_after:.2%}   (gain {yt_acc_after-yt_acc_before:+.2%})")
print(f"UCF base: {ucf_acc_before:.2%} -> {ucf_acc_after:.2%}   (change {ucf_acc_after-ucf_acc_before:+.2%})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
groups = ["YouTube", "UCF base"]
before = [yt_acc_before * 100, ucf_acc_before * 100]
after  = [yt_acc_after * 100,  ucf_acc_after * 100]
x = np.arange(len(groups))

ax.bar(x - 0.2, before, 0.4, label="before")
ax.bar(x + 0.2, after,  0.4, label="after")
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylabel("accuracy (%)")
ax.set_title("Domain adaptation: gain vs forgetting")
ax.legend()

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/stage9_adaptation.png", dpi=150, bbox_inches="tight")
plt.show()

## **Save**

In [ ]:
payload = {
    "stage":          "9_domain_adapt",
    "adapt_arm":      ADAPT_ARM,
    "yt_acc_before":  float(yt_acc_before),
    "yt_acc_after":   float(yt_acc_after),
    "ucf_acc_before": float(ucf_acc_before),
    "ucf_acc_after":  float(ucf_acc_after),
    "skipped":        skipped,
}
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage9_da.json", "w") as f:
    json.dump(payload, f, indent=2, default=float)

torch.save(adapted.state_dict(), f"{cfg.CKPT_DIR}/head_adapted_yt.pt")
print(f"Saved -> {cfg.RESULTS_DIR}/stage9_da.json")